In [ ]:
"""
Optimized ETH Whale Data Pipeline
- Fast incremental loading with proper caching
- All files organized in data/ folder
"""

import os
import time
import json
import requests
import pandas as pd
from datetime import timedelta
from dotenv import load_dotenv

# Setup
load_dotenv()
DUNE_API_KEY = os.getenv("DUNE_WHALES_API")
COINGECKO_API_KEY = os.getenv("COINGECKO_API_KEY")
os.makedirs("data", exist_ok=True)
os.makedirs("data/price_cache", exist_ok=True)

QUERIES = {
    "whales": ("6395391", "data/dune_whales_cache.json", "data/whale_ml_ready.csv"),
    "market_intent": ("6385600", "data/dune_intent_cache.json", "data/market_intent_ml_ready.csv")
}

# ============================================================================
# OPTIMIZED DUNE FETCH (ONLY FETCHES NEW DATA)
# ============================================================================

def fetch_dune(qid, cache):
    """Fetch only new data from Dune, skip if cache is current"""
    headers = {"x-dune-api-key": DUNE_API_KEY}
    today = pd.Timestamp.now(tz='UTC').normalize()
    yesterday = today - timedelta(1)
    
    # Load cache
    if os.path.exists(cache):
        with open(cache) as f:
            c = json.load(f)
        df_cached = pd.DataFrame(c["data"])
        df_cached["block_date"] = pd.to_datetime(df_cached["block_date"], utc=True)
        last_date = pd.to_datetime(c["last_block_date"], utc=True)
        
        # Cache is current - no API call needed
        if last_date >= yesterday:
            print(f"✅ {os.path.basename(cache)} current ({last_date.date()})")
            return df_cached
        
        print(f"🔄 {os.path.basename(cache)}: fetching {(today - last_date).days} new days")
    else:
        df_cached = pd.DataFrame()
        print(f"🆕 {os.path.basename(cache)}: full fetch")
    
    # Execute query ONLY if needed
    resp = requests.post(
        f"https://api.dune.com/api/v1/query/{qid}/execute",
        headers=headers,
        timeout=30
    ).json()
    
    if "execution_id" not in resp:
        raise RuntimeError(f"Dune API error: {resp}")
    
    eid = resp["execution_id"]
    
    # Poll with shorter intervals
    for _ in range(60):  # 10 min max
        status = requests.get(
            f"https://api.dune.com/api/v1/execution/{eid}/status",
            headers=headers
        ).json()["state"]
        
        if status == "QUERY_STATE_COMPLETED":
            break
        if status == "QUERY_STATE_FAILED":
            raise RuntimeError("Query failed")
        time.sleep(10)
    
    # Get results
    result = requests.get(
        f"https://api.dune.com/api/v1/execution/{eid}/results",
        headers=headers
    ).json()["result"]["rows"]
    
    df_new = pd.DataFrame(result)
    if df_new.empty:
        return df_cached
    
    df_new["block_date"] = pd.to_datetime(df_new["block_date"], utc=True)
    
    # Merge with cache
    df = pd.concat([
        df_cached,
        df_new[df_new["block_date"] < today]
    ]).drop_duplicates("block_date", keep="last").sort_values("block_date").reset_index(drop=True)
    
    # Save cache
    with open(cache, "w") as f:
        json.dump({
            "last_block_date": df["block_date"].max().strftime("%Y-%m-%d"),
            "data": json.loads(df.to_json(orient="records", date_format="iso"))
        }, f)
    
    new_rows = len(df_new[df_new["block_date"] < today])
    print(f"✅ {os.path.basename(cache)}: {len(df)} rows (+{new_rows} new)")
    return df

# ============================================================================
# OPTIMIZED COINGECKO FETCH
# ============================================================================

def to_utc(ts):
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")

def fetch_cg_chunked(cg_id, start, end, key=None, days=30):
    """Fetch daily prices from CoinGecko"""
    url = "https://pro-api.coingecko.com/api/v3" if key else "https://api.coingecko.com/api/v3"
    headers = {"x-cg-pro-api-key": key} if key else {}
    
    start_dt, end_dt = to_utc(start), to_utc(end) + pd.Timedelta(days=1)
    all_prices, curr = [], start_dt
    
    while curr < end_dt:
        next_dt = min(curr + pd.Timedelta(days=days), end_dt)
        params = {
            "vs_currency": "usd",
            "from": int(curr.timestamp()),
            "to": int(next_dt.timestamp())
        }
        
        for attempt in range(3):
            try:
                r = requests.get(
                    f"{url}/coins/{cg_id}/market_chart/range",
                    params=params, headers=headers, timeout=30
                )
                r.raise_for_status()
                prices = r.json().get("prices", [])
                all_prices.extend(prices)
                print(f"📥 {cg_id}: {curr.date()} → {next_dt.date()} ({len(prices)} pts)")
                time.sleep(0.3)
                break
            except Exception as e:
                if attempt == 2: raise
                print(f"⚠️ Retry {attempt + 1}/3 ({e})")
                time.sleep(5)
        
        curr = next_dt
    
    if not all_prices:
        return pd.DataFrame(columns=["date", "price"])
    
    # Daily aggregation
    df = pd.DataFrame(all_prices, columns=["timestamp", "price"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.floor("D")
    df = df.groupby("date", as_index=False)["price"].mean().sort_values("date")
    
    # Fill gaps
    full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D", tz="UTC")
    df = df.set_index("date").reindex(full_range).rename_axis("date").reset_index()
    
    return df

def get_price(sym, cg_id, start, end, key=None):
    """Load prices with caching (excludes today)"""
    cache = f"data/price_cache/{sym}.csv"
    
    today_utc = pd.Timestamp.utcnow().floor("D")
    yesterday = today_utc - pd.Timedelta(days=1)
    start, end = to_utc(start), min(to_utc(end), yesterday)
    
    if start > end:
        return pd.DataFrame(columns=["date", f"{sym}_price"])
    
    # Check cache
    if os.path.exists(cache):
        df = pd.read_csv(cache, parse_dates=["date"])
        df["date"] = df["date"].apply(to_utc)
        last_cached = df["date"].max()
        
        if last_cached >= end:
            print(f"✅ {sym.upper()} cache current ({last_cached.date()})")
            return df
        
        fetch_start = last_cached + pd.Timedelta(days=1)
        print(f"🔄 {sym.upper()}: fetching {fetch_start.date()} → {end.date()}")
        
        new = fetch_cg_chunked(cg_id, fetch_start, end, key)
        if not new.empty:
            new = new.rename(columns={"price": f"{sym}_price"})
            df = pd.concat([df, new]).drop_duplicates("date", keep="last").sort_values("date").reset_index(drop=True)
    else:
        print(f"📦 {sym.upper()}: full fetch {start.date()} → {end.date()}")
        df = fetch_cg_chunked(cg_id, start, end, key)
        if not df.empty:
            df = df.rename(columns={"price": f"{sym}_price"})
    
    df.to_csv(cache, index=False)
    print(f"✅ {sym.upper()} saved (through {df['date'].max().date()})")
    return df

# ============================================================================
# MAIN DATA LOADING
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("LOADING WHALE & MARKET DATA")
    print("="*60)
    
    # Load Dune data
    datasets = {}
    for name, (qid, cache, output) in QUERIES.items():
        datasets[name] = fetch_dune(qid, cache)
        datasets[name].to_csv(output, index=False)
        time.sleep(0.5)
    
    df_whales = datasets["whales"]
    df_market_intent = datasets["market_intent"]
    print(f"\n✅ Whales: {len(df_whales)} | Intent: {len(df_market_intent)}")
    
    # Load prices
    print("\n" + "="*60)
    print("LOADING PRICE DATA")
    print("="*60)
    
    min_date = min(df_whales["block_date"].min(), df_market_intent["block_date"].min()) - pd.Timedelta(days=100)
    max_date = max(df_whales["block_date"].max(), df_market_intent["block_date"].max())
    
    print(f"\n📅 Range: {min_date.date()} → {max_date.date()}\n")
    
    df_btc = get_price("btc", "bitcoin", min_date, max_date, COINGECKO_API_KEY)
    df_eth = get_price("eth", "ethereum", min_date, max_date, COINGECKO_API_KEY)
    
    print(f"\n✅ BTC: {df_btc['date'].max().date()} | ETH: {df_eth['date'].max().date()}")

LOADING WHALE & MARKET DATA
🆕 dune_whales_cache.json: full fetch
✅ dune_whales_cache.json: 1095 rows (+1095 new)
🆕 dune_intent_cache.json: full fetch
✅ dune_intent_cache.json: 1095 rows (+1095 new)

✅ Whales: 1095 | Intent: 1095

LOADING PRICE DATA

📅 Range: 2022-09-20 → 2025-12-27

✅ BTC cache current (2025-12-27)
✅ ETH cache current (2025-12-27)

✅ BTC: 2025-12-27 | ETH: 2025-12-27


In [3]:
"""
Compact ETH Whale ML Pipeline
Phase 1: Feature Engineering (NO TARGET)
Phase 2: Target Construction (ISOLATED)
"""

import pandas as pd
import numpy as np

# ============================================================================
# MERGE DATASETS
# ============================================================================

def merge_datasets():
    """Load and merge all datasets"""
    print("📂 Loading datasets...")
    
    df_whales = pd.read_csv('data/whale_ml_ready.csv', parse_dates=['block_date'])
    df_intent = pd.read_csv('data/market_intent_ml_ready.csv', parse_dates=['block_date'])
    df_btc = pd.read_csv('data/price_cache/btc.csv', parse_dates=['date'])
    df_eth = pd.read_csv('data/price_cache/eth.csv', parse_dates=['date'])
    
    # UTC conversion
    for df in [df_whales, df_intent]:
        df['block_date'] = pd.to_datetime(df['block_date'], utc=True)
    for df in [df_btc, df_eth]:
        df['date'] = pd.to_datetime(df['date'], utc=True)
    
    # Merge
    df_prices = pd.merge(df_btc, df_eth, on='date', how='outer').sort_values('date')
    df = pd.merge(df_whales, df_prices, left_on='block_date', right_on='date', how='left').drop(columns=['date'])
    df = pd.merge(df, df_intent, on='block_date', how='left', suffixes=('', '_intent'))
    
    df.to_csv('data/merged_ml_dataset.csv', index=False)
    print(f"✅ Merged: {len(df)} rows, {len(df.columns)} cols")
    return df

# ============================================================================
# PHASE 1: FEATURE ENGINEERING (NO TARGET)
# ============================================================================

def add_features(df, price_col, prefix):
    """Add all price features in one pass"""
    df[f'{prefix}_log_return'] = np.log(df[price_col] / df[price_col].shift(1))
    
    # Lags
    for lag in [1, 3, 7]:
        df[f'{prefix}_log_return_lag{lag}'] = df[f'{prefix}_log_return'].shift(lag)
    
    # Volatility
    df[f'{prefix}_vol7'] = df[f'{prefix}_log_return'].rolling(7, min_periods=1).std()
    df[f'{prefix}_vol30'] = df[f'{prefix}_log_return'].rolling(30, min_periods=1).std()
    
    # RSI
    ret = df[f'{prefix}_log_return']
    gains = ret.where(ret > 0, 0).rolling(14, min_periods=1).mean()
    losses = -ret.where(ret < 0, 0).rolling(14, min_periods=1).mean()
    df[f'{prefix}_rsi'] = 100 - (100 / (1 + gains / (losses + 1e-10)))
    
    # MAs
    df[f'{prefix}_ma7'] = df[price_col].rolling(7, min_periods=1).mean()
    df[f'{prefix}_ma30'] = df[price_col].rolling(30, min_periods=1).mean()
    df[f'{prefix}_price_to_ma7'] = df[price_col] / df[f'{prefix}_ma7']
    df[f'{prefix}_price_to_ma30'] = df[price_col] / df[f'{prefix}_ma30']
    
    return df

def engineer_features(df):
    """PHASE 1: Feature engineering (NO TARGET)"""
    df = df.sort_values('block_date').reset_index(drop=True)
    
    # Price features
    df = add_features(df, 'eth_price', 'eth')
    df = add_features(df, 'btc_price', 'btc')
    
    # ETH-BTC features
    df['eth_btc_ratio'] = df['eth_price'] / df['btc_price']
    df['eth_btc_ratio_ma7'] = df['eth_btc_ratio'].rolling(7, min_periods=1).mean()
    df['eth_btc_ratio_ma30'] = df['eth_btc_ratio'].rolling(30, min_periods=1).mean()
    df['eth_btc_corr_30d'] = df['eth_log_return'].rolling(30, min_periods=20).corr(df['btc_log_return'])
    df['eth_outperformance'] = df['eth_log_return'] - df['btc_log_return']
    df['eth_outperformance_ma7'] = df['eth_outperformance'].rolling(7, min_periods=1).mean()
    
    df.to_csv('data/features_engineered.csv', index=False)
    print(f"✅ Features: {len(df.columns)} cols, {len(df)} rows")
    return df

# ============================================================================
# PHASE 2: TARGET CONSTRUCTION (ISOLATED)
# ============================================================================

def create_target(df):
    """PHASE 2: Target creation (ISOLATED from Phase 1)"""
    df = df.sort_values('block_date').reset_index(drop=True)
    
    # Create target
    df["next_day_return"] = df["eth_log_return"].shift(-1)
    df["target"] = (df["next_day_return"] > 0).astype(int)
    
    # Remove leakage
    X = df.drop(columns=['target', 'next_day_return', 'eth_log_return', 'btc_log_return'])
    y = df['target']
    
    # Remove NaN rows
    valid = ~y.isna()
    X, y = X[valid], y[valid]
    
    # Save
    ml_ready = X.copy()
    ml_ready['target'] = y
    ml_ready.to_csv('data/ml_ready_dataset.csv', index=False)
    
    # Stats
    print(f"✅ X: {X.shape}, y: {y.shape}")
    print(f"   Up: {(y==1).sum()} ({(y==1).sum()/len(y)*100:.1f}%)")
    print(f"   Down: {(y==0).sum()} ({(y==0).sum()/len(y)*100:.1f}%)")
    
    return X, y

# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    # Merge
    df = merge_datasets()
    
    # Phase 1: Features
    print("\n" + "="*60)
    print("PHASE 1: FEATURE ENGINEERING")
    print("="*60)
    df_feat = engineer_features(df)
    
    # Phase 2: Target
    print("\n" + "="*60)
    print("PHASE 2: TARGET CONSTRUCTION")
    print("="*60)
    X, y = create_target(df_feat)
    
    print("\n✅ Pipeline complete - ready for modeling!")

📂 Loading datasets...
✅ Merged: 1095 rows, 36 cols

PHASE 1: FEATURE ENGINEERING
✅ Features: 64 cols, 1095 rows

PHASE 2: TARGET CONSTRUCTION
✅ X: (1095, 62), y: (1095,)
   Up: 572 (52.2%)
   Down: 523 (47.8%)

✅ Pipeline complete - ready for modeling!
